In [1]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../data/processed/icu_mortality_cohort_demo.csv")

modeling_df = pd.read_csv(DATA_PATH)

print(modeling_df.shape)
modeling_df.head()

(128, 12)


,subject_id,hadm_id,stay_id,gender,anchor_age,admission_type,admission_location,insurance,marital_status,race,first_careunit,hospital_expire_flag
0,10023771,20044587,33177122,M,70,ELECTIVE,PHYSICIAN REFERRAL,Medicare,MARRIED,WHITE,Cardiac Vascular Intensive Care Unit (CVICU),0
1,10005909,20199380,36496303,F,40,OBSERVATION ADMIT,EMERGENCY ROOM,Other,MARRIED,WHITE,Cardiac Vascular Intensive Care Unit (CVICU),0
2,10003400,20214994,32128372,F,72,URGENT,TRANSFER FROM SKILLED NURSING FACILITY,Medicare,MARRIED,BLACK/AFRICAN AMERICAN,Medical/Surgical Intensive Care Unit (MICU/SICU),0
3,10008454,20291550,31959184,F,26,EW EMER.,EMERGENCY ROOM,Other,SINGLE,WHITE,Trauma SICU (TSICU),0
4,10019385,20297618,39268883,M,44,URGENT,TRANSFER FROM HOSPITAL,Other,MARRIED,WHITE,Cardiac Vascular Intensive Care Unit (CVICU),0


In [ ]:
modeling_df.info()   
# Display information about the DataFrame
# This includes the number of rows, columns, data types, and memory usage, and if there's missing data in any of the columns.

In [ ]:
modeling_df.describe()
#only summarize the numeric columns in the DataFrame. 
#  It provides statistics such as count, mean, standard deviation, minimum, maximum, and quartiles (25%, 50%, 75%) for each numeric column.

In [ ]:
modeling_df.isnull().sum()
#how many missing values are present in each column of the DataFrame.

In [4]:
(modeling_df.isnull().mean()*100).round(2)
#calculate the mean of missing value (NaN) for each column, 
# multiply by 100 to get the percentage, and round to two decimal places.

subject_id              0.00
hadm_id                 0.00
stay_id                 0.00
gender                  0.00
anchor_age              0.00
admission_type          0.00
admission_location      0.00
insurance               0.00
marital_status          7.81
race                    0.00
first_careunit          0.00
hospital_expire_flag    0.00
dtype: float64

In [ ]:
import missingno as msno

msno.matrix(modeling_df)

In [5]:
X = modeling_df.drop(
    columns=[
        "subject_id",
        "hadm_id",
        "stay_id",
        "hospital_expire_flag"
    ]
)

y = modeling_df["hospital_expire_flag"]

In [6]:
X.head()

,gender,anchor_age,admission_type,admission_location,insurance,marital_status,race,first_careunit
0,M,70,ELECTIVE,PHYSICIAN REFERRAL,Medicare,MARRIED,WHITE,Cardiac Vascular Intensive Care Unit (CVICU)
1,F,40,OBSERVATION ADMIT,EMERGENCY ROOM,Other,MARRIED,WHITE,Cardiac Vascular Intensive Care Unit (CVICU)
2,F,72,URGENT,TRANSFER FROM SKILLED NURSING FACILITY,Medicare,MARRIED,BLACK/AFRICAN AMERICAN,Medical/Surgical Intensive Care Unit (MICU/SICU)
3,F,26,EW EMER.,EMERGENCY ROOM,Other,SINGLE,WHITE,Trauma SICU (TSICU)
4,M,44,URGENT,TRANSFER FROM HOSPITAL,Other,MARRIED,WHITE,Cardiac Vascular Intensive Care Unit (CVICU)


In [7]:
y.head()

0    0
1    0
2    0
3    0
4    0
Name: hospital_expire_flag, dtype: int64

In [8]:
X.isnull().sum()

gender                 0
anchor_age             0
admission_type         0
admission_location     0
insurance              0
marital_status        10
race                   0
first_careunit         0
dtype: int64

In [32]:
categorical_columns = X.select_dtypes(include="object").columns
# select columns with data type "object" (categorical columns) from the DataFrame X
X[categorical_columns] = X[categorical_columns].fillna("Unknown")
# fill missing values in categorical columns with "Unknown"

In [33]:
numeric_columns = X.select_dtypes(include=["int64","float64"]).columns

X[numeric_columns] = X[numeric_columns].fillna(
    X[numeric_columns].median()
)

In [12]:
X.isnull().sum()

gender                0
anchor_age            0
admission_type        0
admission_location    0
insurance             0
marital_status        0
race                  0
first_careunit        0
dtype: int64

In [23]:
X = pd.get_dummies(
    X,
    drop_first=True
)
# perform one-hot encoding on categorical variables, dropping the first category to avoid multicollinearity

In [24]:
X.head()

,anchor_age,gender_M,admission_type_ELECTIVE,admission_type_EW EMER.,admission_type_OBSERVATION ADMIT,admission_type_SURGICAL SAME DAY ADMISSION,admission_type_URGENT,admission_location_EMERGENCY ROOM,admission_location_INFORMATION NOT AVAILABLE,admission_location_PACU,...,race_WHITE - BRAZILIAN,race_WHITE - OTHER EUROPEAN,first_careunit_Coronary Care Unit (CCU),first_careunit_Medical Intensive Care Unit (MICU),first_careunit_Medical/Surgical Intensive Care Unit (MICU/SICU),first_careunit_Neuro Intermediate,first_careunit_Neuro Stepdown,first_careunit_Neuro Surgical Intensive Care Unit (Neuro SICU),first_careunit_Surgical Intensive Care Unit (SICU),first_careunit_Trauma SICU (TSICU)
0,70,True,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,40,False,False,False,True,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
2,72,False,False,False,False,False,True,False,False,False,...,False,False,False,False,True,False,False,False,False,False
3,26,False,False,True,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,True
4,44,True,False,False,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [25]:
print(X.shape)

(128, 41)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,  # USE 20% of the data for testing
    random_state=42,  # Every time it starts from seed 42, it produces the exact same shuffle.That's what random_state does.
    stratify=y # Its goal is to 
    #make the class distribution in the training set and test set approximately the same as the original dataset.
)

In [27]:
print(X_train.shape)
print(X_test.shape)

(102, 41)
(26, 41)


In [30]:
print(y_train.value_counts())

hospital_expire_flag
0    90
1    12
Name: count, dtype: int64


In [29]:
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

hospital_expire_flag
0    0.882353
1    0.117647
Name: proportion, dtype: float64
hospital_expire_flag
0    0.884615
1    0.115385
Name: proportion, dtype: float64


In [ ]:
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(exist_ok=True)
#make sure the directory exists, and if it doesn't, create it.

In [ ]:
X_train.to_csv(PROCESSED_DIR/"X_train.csv", index=False)
X_test.to_csv(PROCESSED_DIR/"X_test.csv", index=False)

y_train.to_csv(PROCESSED_DIR/"y_train.csv", index=False)
y_test.to_csv(PROCESSED_DIR/"y_test.csv", index=False)

## Day 3 Summary

- Removed identifier columns
- Examined missing values
- Filled missing data
- Applied one-hot encoding
- Split the data into training and testing sets
- Created machine-learning-ready datasets